# ms2Topo example workflow

This notebook runs ms2Topo on a small example dataset and shows the resulting feature table and consensus spectra.

In [1]:
import sys
import os
home=os.getcwd()
sys.path.append(home+'/Functions')
from ShowDF import *
import numpy as np
import pandas as pd
import sys

## 1. Extract MS2 spectra

This step reads the example MS2 summary files and exports individual MS2 spectra into a folder structure that ms2Topo can later query.

For the example, the extraction is restricted to a small m/z and RT window so the notebook runs quickly.

In [2]:
from BatchExtract_All_MS2_Spectra import *

DataFolder = home + '/Playground/Data'
saveFolder = home + '/Playground/ms2_spectra-20260520'

BatchExtract_All_MS2_Spectra(DataFolder = DataFolder,
                             saveFolder = saveFolder,
                             min_RT = 0,
                             max_RT = 1250,
                             min_mz = 245,
                             max_mz = 250,
                             minPeaks = 1,
                             SFindicator = '-ms2Summary.csv')

#Spectra that needed to and could be picked by MS-level:
  MS-level 2: 1 / 1


## 2. Join MS2 summary tables

The extracted sample-level MS2 summaries are joined into one table.  
This table is the starting point for precursor-level grouping before MS2-topological refinement.

In [6]:
from JoiningSummMS2 import *

# for reading the already extracted ms2-spectra, but you can also extract them again or use your own data
ms2Folder = home + '/Playground/ms2_spectra-20260520'

All_SummMS2Table, SamplesNames = JoiningSummMS2(ResultsFolder = ms2Folder,
                                                mz_min = 245, 
                                                mz_max = 250,
                                                RT_min = 0,
                                                RT_max = 1250,
                                                ToReplace = 'mzML-ms2Summary.xlsx')

print("Joined MS2 summary table shape:", All_SummMS2Table.shape)
print("Number of samples:", len(SamplesNames))
print("First samples:", SamplesNames[:5])

Joined MS2 summary table shape: (2645, 8)
Number of samples: 24
First samples: ['A', 'B', 'C', 'D', 'E']


## 3. Define m/z workload slices

ms2Topo processes the data in m/z slices so large datasets can be handled in manageable chunks.
For this small example, the slicing is mainly a convenient way to keep the workflow modular.

In [7]:
from WorkLoadPlanning import *

EdgesMat = WorkLoadPlanning(All_SummMS2Table = All_SummMS2Table,
                            slices = 1,
                            mz_Tol = 2e-3)
print("Number of m/z slices:", len(EdgesMat))
print(EdgesMat)

Number of m/z slices: 1
[[   0 2645    0]]


## 4. Set workflow parameters

The parameters below define how ms2Topo retrieves spectra, groups precursor features, aligns fragments, clusters MS2 similarity networks, and closes final consensus features.

The values are chosen for this example dataset and are not universal defaults.

In [5]:
from make_ms2topo_context import *

ProjectName = 'ms2Topo_Features_table'

params = {"io": {"ms2Folder": ms2Folder,
                 "ToAdd": "mzML",
                 "Norm2One": True},
          
          "columns": {"mz_col": 1,
                      "RT_col": 2,
                      "sample_id_col": 6,
                      "ms2_spec_id_col": 0},
          # Initial precursor-level grouping before MS2 refinement
          "feature_grouping": {"RT_tol": 30,
                               "mz_Tol": 5e-4},
          # Retain fragments explaining most of the spectrum intensity
          "alignment": {"Intensity_to_explain": 0.95,
                        "min_spectra_fraction": 0.3},
          # Repeated subsampling is used to estimate a plausible number of MS2 modules
          "clustering": {"max_Nspectra_cluster": 8,
                         "Nspectra_sampling": 50,
                         "SamplingTimes": 20,
                         "std_times": 1,
                         "min_nodes": 1,
                         "assign_labels": "discretize",
                         "random_state": 0},

           "summary": {"percentile": 10,
                       "percentile_mz": 5,
                       "percentile_Int": 10,
                       "percentile_RT": 5},

           "closing": {"min_spectra": 3,
                       "minSpectra": 3,
                       "alpha": 0.01}}

context = make_ms2topo_context(ProjectName = ProjectName,
                               All_SummMS2Table = All_SummMS2Table,
                               EdgesMat = EdgesMat,
                               SamplesNames = SamplesNames,
                               feature_id = 0)

## 5. Run ms2Topo feature extraction

This cell runs the full MS2-topology workflow on the example data.

For each precursor-level group, ms2Topo retrieves the associated MS2 spectra, aligns fragments, builds a cosine-similarity network, partitions spectra into MS2-consistent modules, and writes final feature/consensus outputs.

Generates Alignedms2Features(date), a folder with all the consensus spectra as well as the information to track them back, and a .csv file containing the features table.

In [6]:
from ms2_SamplesAligment import *


AlignedSamplesDF = ms2_SamplesAligment(context = context,
                                       params = params)

### ms2Topo-features-table

The output table contains one row per final ms2Topo feature object.
The columns summarize precursor statistics, MS2 module quality, and sample-level presence/absence.

In [8]:
print("Aligned feature table shape:", AlignedSamplesDF.shape)
ShowDF(AlignedSamplesDF)

Aligned feature table shape: (33, 46)


,median_mz(Da),IQR_mz(ppm),Q1_RT(s),median_RT(s),Q3_RT(s),N_samples,N_ms2-spectra,Q1_silhouette,median_silhouette,Q3_silhouette,Q1_intramodule_cosine,median_intramodule_cosine,Q3_intramodule_cosine,min_mz,max_mz,min_RT(s),max_RT(s),min_silhouette,max_silhouette,min_intramodule_cosine,max_intramodule_cosine,feat_id,A,B,C,D,E,F,G,H,I,J,K,L,M,N,O,P,Q,R,S,T,U,V,W,X
0,245.078,0.469523,217.766,221.655,223.356,18,32,0.449316,0.773319,0.782051,0.592923,0.698322,0.980971,245.078,245.078,216.343,227.687,0.260347,0.785965,0.448473,0.990516,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,1,1,1,1,1,1,1,1,1,0
1,245.078,0.566613,217.803,221.61,223.204,8,15,0.0647523,0.470528,0.550395,0.241007,0.624045,0.69533,245.078,245.078,216.64,228.472,-0.0529293,0.563643,0,0.818401,1,0,0,0,0,1,1,0,0,0,0,0,1,1,1,0,1,1,0,0,0,0,0,0,1
2,245.113,0.653647,44.9901,54.3022,65.1552,2,8,0.953222,0.955207,0.958274,0.947487,0.956005,0.962086,245.113,245.113,32.8071,72.8667,0.948887,0.959358,0.940694,0.964579,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0
3,245.115,2.36555,179.859,181.269,183.001,6,7,0.996955,0.997313,0.997738,0.995533,0.99849,0.998974,245.115,245.116,179.644,184.838,0.995101,0.998049,0.99138,0.999106,3,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0
4,245.126,0.314696,128.346,129.275,129.972,8,8,0.893642,0.918891,0.921332,0.883059,0.909996,0.947825,245.126,245.126,128.025,130.126,0.876697,0.927664,0.842341,0.971932,4,0,0,0,0,0,0,0,1,1,0,0,1,1,1,1,0,1,0,0,0,0,0,0,1
5,245.138,0.560765,105.052,213.997,441.594,24,1199,0.855704,0.895341,0.907322,0.777616,0.939029,0.977668,245.138,245.138,24.3733,521.535,0.671727,0.909983,0.497927,0.990967,5,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
6,245.186,0.560102,87.028,87.534,88.3131,3,3,0.800992,0.826356,0.833434,0.76855,0.789785,0.865876,245.186,245.186,86.6233,88.9363,0.785774,0.837681,0.761472,0.89124,6,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
7,245.229,0.567988,432.893,469.318,503.14,24,618,0.244655,0.605626,0.650628,0,0.419828,0.892883,245.229,245.23,405.838,531.055,4.32037e-05,0.658139,0,0.975758,7,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
8,245.229,0.49778,15.8165,39.1808,55.2801,24,264,0.00051392,0.358589,0.486089,0,0,0.607265,245.229,245.229,4.42675,72.0815,0,0.503598,0,0.842491,8,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
9,246.142,0.743903,439.734,471.451,504.448,6,65,0.927592,0.94219,0.951928,0.921852,0.942419,0.959242,246.141,246.142,411.128,532.723,0.908237,0.957705,0.900548,0.971661,9,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,1,0,0


## 6. Inspect consensus spectra

For each final feature module, ms2Topo writes a consensus MS2 spectrum.  
These spectra summarize the reproducible fragments supporting each MS2-topology feature.

In [26]:
from pathlib import Path
import pandas as pd

consensus_folders = home + '/Alignedms2Features2026-05-20'

consensus_files = [x for x in os.listdir(consensus_folders)               
                   if x.startswith('Consensus_ms2-spectra_')]
consensus_files.sort()

print("Number of consensus spectra:", len(consensus_files))

example_consensus = pd.read_csv(consensus_folders + '/' + consensus_files[0],
                                index_col=0)
ShowDF(example_consensus)

Number of consensus spectra: 33


,median_mz(Da),N_spectra,min_mz,max_mz,IQR_mz(ppm),mean_Int,min_Int,max_Int,median_Int,Int_Q1,Int_Q3
0,55.0542,20,55.054,55.0544,3.02607,5.38121,0,9.96932,7.04196,0,8.58289
1,57.0698,6,57.0697,57.07,1.99863,2.48837,0,4.29039,0,0,0
2,59.0491,13,59.049,59.0492,1.22169,1.79265,0,4.80623,0,0,4.15472
3,74.0601,4,74.0601,74.0602,1.68314,0.873101,0,0.927655,0,0,0
4,81.0699,5,81.0697,81.0702,1.62501,0.340593,0,0.868762,0,0,0
5,83.049,21,83.0489,83.0494,2.88547,8.7791,0,15.0613,11.9555,0,13.7773
6,85.0284,10,85.0282,85.0285,1.96559,1.08292,0,3.55932,0,0,3.02658
7,85.0647,5,85.0642,85.0651,0.963442,1.03295,0,1.00028,0,0,0
8,95.0854,17,95.0852,95.0858,3.22112,3.04819,0,6.5252,3.93597,0,5.48533
9,97.1009,4,97.1009,97.1013,1.61901,0.127743,0,0.829246,0,0,0
